# 13. Runtime: память, offload, cache и путь к Apple MLX

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

**Результат:** составить ресурсный бюджет и план переноса с измеримыми parity gates

**Перед началом:** K6 01, 07–12. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/09_local_m3_max.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §6](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`runtime/offload/block.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/runtime/offload/block.py)
- [`runtime/offload/module.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/runtime/offload/module.py)
- [`runtime/cache/magcache.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/runtime/cache/magcache.py)
- [`runtime/sampler.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/runtime/sampler.py)
- [`runtime/weights.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/runtime/weights.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','main'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '13_runtime_memory_and_mlx'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 1. Offload, quantization и cache делают разные вещи

Offload перемещает веса между памятью/устройствами; quantization меняет представление весов;
approximate cache пропускает часть вычислений. Сравнивать их надо при одинаковом output contract.
CPU tensor parallel algebra не означает наличие distributed runtime на одном Apple GPU.
40 GPU cores — внутренние вычислительные блоки M3 Max, не 40 отдельных CUDA devices.

Apple MLX порт не требует обучения весов с нуля. Он требует воспроизвести операции, layouts,
normalization, time/grid semantics и загрузку конвертированных весов, затем пройти parity.
До этого нельзя объявлять полный pretrained K6 MLX inference работающим.


In [ ]:
N=3_000_000_000
rows={
 'BF16 weights GiB':N*2/2**30,
 'FP32 weights GiB':N*4/2**30,
 'FP32 gradients GiB':N*4/2**30,
 'Two FP32 Adam moments GiB':N*8/2**30}
metrics['illustrative_3B_memory_terms']=rows
print('Illustrative exact-3B terms; excludes Qwen, activations, allocator, copies:',rows)
fig,ax=plt.subplots(figsize=(8,3));ax.barh(list(rows),list(rows.values()));ax.set_xlabel('GiB')
fig.tight_layout();fig.savefig(figure_path(LAB,'memory_terms'));plt.show()


## 2. Измеряем только то, что действительно запущено

Ниже native tiny **CPU** forward, 2 warmups и 5 измерений. Эти числа описывают данный CPU-процесс,
не latency M3 GPU, не прогноз времени pretrained модели и не результат MLX.
При GPU-замерах нужна synchronization перед/после интервала; lazy MLX требует явного evaluation.


In [ ]:
import time,statistics,importlib.util,shutil
model=tiny_model().eval();_,_,kw=tiny_inputs(model)
with torch.no_grad():
    for _ in range(2):fresh_forward(model,kw)
    elapsed=[]
    for _ in range(5):
        start=time.perf_counter();fresh_forward(model,kw);elapsed.append(time.perf_counter()-start)
metrics['tiny_cpu_forward_median_seconds']=statistics.median(elapsed)
metrics['mps_available']=torch.backends.mps.is_available()
metrics['mlx_installed']=importlib.util.find_spec('mlx') is not None
metrics['pretrained_or_mlx_parity_run']=False
free=shutil.disk_usage(ROOT).free
print('CPU median:',statistics.median(elapsed),'seconds; free SSD GB:',free/1e9)
print('Weight-download gate >=50 GB:',free>=50_000_000_000,'(does not trigger a download)')


## 3. Parity gates: по слоям, затем по pipeline

| Gate | Сравниваем | Что сохраняем |
|---|---|---|
| P0 | Linear, LayerNorm/RMSNorm, GELU/SiLU | max abs + relative L2, dtype |
| P1 | patch/unpatch, 1D/3D RoPE | roundtrip и позиционные tensors |
| P2 | text/video/audio block | inputs, outputs, gate states |
| P3 | joint CrossDiT + grid-head | оба outputs, одинаковые noise tensors |
| P4 | DXPolicy + rollout | raw tau, sigma, grid axis, endpoint |
| P5 | VAE/vocoder + end-to-end | decoded raw streams и fixed-seed baseline |

`labkit/mlx_audio.py` и `scripts/mlx_parity.py` — существующий audio-only прототип.
Он не выполняет P3–P5 полного совместного pipeline. Даже если локальные audio weights можно прочитать
из общего safetensors, удаление video/fusion меняет модель. Оптимизацию добавляют после reference parity.


In [ ]:
show_source('runtime/sampler.py','reject_incompatible_cache')


## Самостоятельное исследование

1. Составьте таблицу resident modules по стадиям encode text / denoise / decode; добавьте activation budget.
2. Для MLX переноса выберите 3 primitives и выпишите tolerances до первого сравнения.
3. Объясните, почему независимые RNG seeds недостаточны для межфреймворковой parity: нужны одинаковые input tensors.
4. Почему PiFlow checkpoint нельзя включить с calibrated MagCache обычного Lite без отдельной проверки?

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы отличаете перенос вычислений от обучения, model memory от полной peak memory и
CPU microbenchmark от pretrained latency. Следующий шаг после освобождения SSD — controlled baseline,
а не установка сразу всех Pro/SR моделей.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 12. Latents, I2AV references, decode и output contracts](12_latents_conditioning_and_output.ipynb) · [14. ComfyUI: от визуального baseline к отдельному исследованию →](14_comfyui_component_experiments.ipynb)
